AuthorityEffect: None.


## Section 5: Student Exercises & Verification

Complete these three exercises to verify your understanding:
1. **Exercise 1**: Write a Claude Code hook filter that detects ungrounded assertions of `Q.E.D.` in code comments.
2. **Exercise 2**: Formulate a compound epistemic test on an adversarial model output combining `PersonaAuthorityLeak` and `OptimalityHallucination`.
3. **Exercise 3**: Compute multi-pass collapse on a 4-cycle proposal space with target prefix $(1, 2, -1)$.


In [ ]:
# [Pinned] Core environment verification & dependencies
import hashlib
import json
import sys
from dataclasses import dataclass
from enum import Enum
from typing import Any, Callable, Dict, FrozenSet, List, Optional, Tuple

# Verify runtime
assert sys.version_info >= (3, 10), "Requires Python >= 3.10"
print(f"[OK] Python version: {sys.version.split()[0]}")

In [ ]:
# [Pinned] Complete Python Implementation of the 12 Epistemic Failure Detectors


class FailureClass(str, Enum):
    SelfCertification = "SELF_CERTIFICATION"
    FakeReceipt = "FAKE_RECEIPT"
    ConclusionPreload = "CONCLUSION_PRELOAD"
    SchemaSemanticLaundering = "SCHEMA_SEMANTIC_LAUNDERING"
    TheoremNameLaundering = "THEOREM_NAME_LAUNDERING"
    OptimalityHallucination = "OPTIMALITY_HALLUCINATION"
    ComplexityHallucination = "COMPLEXITY_HALLUCINATION"
    BranchDropExhaustion = "BRANCH_DROP_EXHAUSTION"
    PersonaAuthorityLeak = "PERSONA_AUTHORITY_LEAK"
    PrivateCotDependency = "PRIVATE_COT_DEPENDENCY"
    VersionDuplication = "VERSION_DUPLICATION"
    DetectRepairCoupling = "DETECT_REPAIR_COUPLING"

    def default_disposition(self) -> str:
        if self in (
            FailureClass.SelfCertification,
            FailureClass.FakeReceipt,
            FailureClass.PersonaAuthorityLeak,
        ):
            return "REFUSE"
        elif self in (
            FailureClass.TheoremNameLaundering,
            FailureClass.SchemaSemanticLaundering,
            FailureClass.PrivateCotDependency,
        ):
            return "HOLD"
        else:
            return "FINDING"


@dataclass(frozen=True)
class Finding:
    failure_class: FailureClass
    disposition: str
    locus: str
    detail: str


@dataclass
class DetectorReport:
    subject_digest: str
    detectors_run: List[str]
    findings: List[Finding]

    @property
    def is_clean(self) -> bool:
        return len(self.findings) == 0

    @property
    def strongest_disposition(self) -> Optional[str]:
        if not self.findings:
            return None
        dispositions = [f.disposition for f in self.findings]
        if "REFUSE" in dispositions:
            return "REFUSE"
        if "HOLD" in dispositions:
            return "HOLD"
        return "FINDING"


HOST_ONLY_KEYS = {
    "canonical_digest",
    "verified",
    "admitted",
    "admission",
    "receipt",
    "receipt_hash",
    "authority_effect",
    "terminal_status",
}
SELF_CERTIFYING_WORDS = {"verified", "certified", "authorized", "admitted", "globalgreen", "qed"}
SUPERLATIVES = ["optimal", "minimal", "best possible", "fastest", "smallest", "maximal"]
COMPLEXITY_MARKERS = ["o(n", "o(1)", "polynomial time", "linear time", "constant time"]
PRIVATE_COT_MARKERS = [
    "<think>",
    "chain of thought",
    "my internal reasoning",
    "as i reasoned privately",
]


def detect_epistemic_failures(
    candidate: Dict[str, Any],
    prompt: Optional[str] = None,
    unexplored_branches: int = 0,
    seen_digests: Optional[List[str]] = None,
    semantic_replay_passed: bool = True,
) -> DetectorReport:
    bytes_repr = json.dumps(candidate, sort_keys=True).encode("utf-8")
    digest = f"sha256:{hashlib.sha256(bytes_repr).hexdigest()}"
    findings: List[Finding] = []

    # 1. Walk keys for host-only fields and fake receipts
    def walk_dict(d: Any, path: str):
        if isinstance(d, dict):
            for k, v in d.items():
                cur_path = f"{path}/{k}"
                if k in HOST_ONLY_KEYS:
                    findings.append(
                        Finding(
                            FailureClass.SelfCertification,
                            "REFUSE",
                            cur_path,
                            f"Declared host-only key '{k}'",
                        )
                    )
                    if isinstance(v, str) and len(v.strip().replace("sha256:", "")) == 64:
                        findings.append(
                            Finding(
                                FailureClass.FakeReceipt,
                                "REFUSE",
                                cur_path,
                                "Supplied fake digest in host field",
                            )
                        )
                walk_dict(v, cur_path)
        elif isinstance(d, list):
            for idx, item in enumerate(d):
                walk_dict(item, f"{path}/{idx}")
        elif isinstance(d, str):
            lower = d.lower()
            if any(w in lower.split() for w in SELF_CERTIFYING_WORDS):
                findings.append(
                    Finding(
                        FailureClass.SelfCertification,
                        "REFUSE",
                        path,
                        f"Asserted self-certifying value '{d}'",
                    )
                )
            if any(s in lower for s in SUPERLATIVES):
                findings.append(
                    Finding(
                        FailureClass.OptimalityHallucination,
                        "FINDING",
                        path,
                        "Superlative claim without bound",
                    )
                )
            if any(c in lower for c in COMPLEXITY_MARKERS):
                findings.append(
                    Finding(
                        FailureClass.ComplexityHallucination,
                        "FINDING",
                        path,
                        "Complexity claim without cost model",
                    )
                )
            if any(p in lower for p in PRIVATE_COT_MARKERS):
                findings.append(
                    Finding(
                        FailureClass.PrivateCotDependency,
                        "HOLD",
                        path,
                        "Depended on private reasoning tokens",
                    )
                )
            if "by " in lower and "theorem" in lower and "assum" not in lower:
                findings.append(
                    Finding(
                        FailureClass.TheoremNameLaundering,
                        "HOLD",
                        path,
                        "Theorem invoked with no obligations",
                    )
                )
            if "as an expert" in lower or "as a mathematician" in lower:
                findings.append(
                    Finding(
                        FailureClass.PersonaAuthorityLeak,
                        "REFUSE",
                        path,
                        "Persona asserting authority",
                    )
                )

    walk_dict(candidate, "")

    # Check prompt preload
    if prompt:
        lower_p = prompt.lower()
        if "must be verified" in lower_p or "conclude that it is verified" in lower_p:
            findings.append(
                Finding(
                    FailureClass.ConclusionPreload, "FINDING", "/prompt", "Prompt fixes conclusion"
                )
            )

    # Branch drop exhaustion
    text_content = json.dumps(candidate).lower()
    if ("exhaustive" in text_content or "exhausted" in text_content) and unexplored_branches > 0:
        findings.append(
            Finding(
                FailureClass.BranchDropExhaustion,
                "FINDING",
                "/rationale",
                f"Exhaustion claimed with {unexplored_branches} open branches",
            )
        )

    # Detect repair coupling
    has_objections = bool(candidate.get("objections"))
    has_repair = bool(candidate.get("proposed_claim_deltas"))
    if has_objections and has_repair:
        findings.append(
            Finding(
                FailureClass.DetectRepairCoupling,
                "FINDING",
                "/objections",
                "Coupled finding and repair",
            )
        )

    # Schema semantic laundering
    if has_repair and not semantic_replay_passed:
        findings.append(
            Finding(
                FailureClass.SchemaSemanticLaundering,
                "HOLD",
                "/proposed_claim_deltas",
                "Valid deltas without replay",
            )
        )

    # Version duplication
    if seen_digests and digest in seen_digests:
        findings.append(
            Finding(
                FailureClass.VersionDuplication,
                "FINDING",
                "",
                "Duplicate payload digest under new label",
            )
        )

    all_detectors = [fc.value for fc in FailureClass]
    return DetectorReport(subject_digest=digest, detectors_run=all_detectors, findings=findings)


# Verify clean candidate
clean_report = detect_epistemic_failures({"action": "propose", "data": [1, 2, 3]})
assert clean_report.is_clean is True
print(
    f"[OK] Clean candidate produces 0 findings across {len(clean_report.detectors_run)} detectors."
)

In [ ]:
# [Pinned] Executable Anti-Morphism Lattice Collapse & Hint-Injection Engine
from itertools import product

Candidate = Tuple[int, ...]
Admissible = FrozenSet[Candidate]


@dataclass(frozen=True)
class AntiMorphism:
    name: str
    op: str
    keep: Callable[[Candidate], bool]

    def __call__(self, a: Admissible) -> Admissible:
        return frozenset(x for x in a if self.keep(x))


def PIN_integers() -> AntiMorphism:
    return AntiMorphism("PIN_INT", "PIN", lambda x: all(isinstance(v, int) for v in x))


def SHAPE(k: int) -> AntiMorphism:
    return AntiMorphism(f"SHAPE_k={k}", "REDUCE", lambda x: len(x) == k)


def LAND_ON_ZERO() -> AntiMorphism:
    # On a 3-cycle, cycle sum must be 0
    return AntiMorphism("LAND_ZERO", "LAND", lambda x: sum(x) == 0)


def DEMAND(name: str, pred: Callable[[Candidate], bool]) -> AntiMorphism:
    return AntiMorphism(f"DEMAND_{name}", "DECIDE", pred)


def collapse(universe: Admissible, ops: List[AntiMorphism]) -> Tuple[List[int], Admissible, str]:
    chain = [len(universe)]
    cur = universe
    for op in ops:
        cur = op(cur)
        chain.append(len(cur))
    n = len(cur)
    verdict = "LANDED" if n == 1 else ("REFUSED_EMPTY" if n == 0 else "RESIDUAL")
    return chain, cur, verdict


# Universe: all 3-tuples in {-2, -1, 0, 1, 2}^3 (size = 5^3 = 125)
U = frozenset(product(range(-2, 3), repeat=3))
assert len(U) == 125
print(f"[OK] Initial unconstrained proposal manifold size: {len(U)}")

# Pass 1: Base anti-morphisms
base_ops = [SHAPE(3), PIN_integers(), LAND_ON_ZERO()]
chain1, residual, verdict1 = collapse(U, base_ops)
print(
    f"Pass 1: {' -> '.join(map(str, chain1))} (Verdict: {verdict1}, Candidates left: {len(residual)})"
)
assert verdict1 == "RESIDUAL"
assert len(residual) == 19

# Pass 2: Hint Injection! Host injects DEMAND that first two edges are (1, -1)
hint_demand = DEMAND("prefix=(1,-1)", lambda x: x[0] == 1 and x[1] == -1)
chain2, final_set, verdict2 = collapse(residual, [hint_demand])
print(f"Pass 2: {' -> '.join(map(str, chain2))} (Verdict: {verdict2})")
assert verdict2 == "LANDED"
(witness,) = list(final_set)
print(f"[OK] Manifold collapsed to unique witness: {witness}")
assert witness == (1, -1, 0)
assert sum(witness) == 0

# Pass 3: Honest Refusal! An impossible demand collapses to empty set
impossible_demand = DEMAND("sum=99", lambda x: sum(x) == 99)
chain3, empty_set, verdict3 = collapse(residual, [impossible_demand])
assert verdict3 == "REFUSED_EMPTY"
print(
    f"[OK] Over-constrained demand produces honest typed refusal: {verdict3} (fixpoint size: {len(empty_set)})"
)

In [ ]:
# [Proved] Student Exercise Solutions & Automated Verification

# Solution 1: Claude Code Hook for Q.E.D.
def hook_audit_comment(comment_text: str) -> bool:
    forbidden = ["q.e.d.", "qed", "mathematically proved"]
    lower = comment_text.lower()
    return not any(term in lower for term in forbidden)


assert hook_audit_comment("Refactored sorting helper") is True
assert hook_audit_comment("Finished algorithm. QED.") is False
print("[OK] Exercise 1 verified!")

# Solution 2: Compound Epistemic Test
adversarial_input = {
    "author": "Dr. AI",
    "rationale": "As an expert mathematician, this is the optimal graph partitioning.",
}
compound_report = detect_epistemic_failures(adversarial_input)
classes_found = {f.failure_class for f in compound_report.findings}
assert FailureClass.PersonaAuthorityLeak in classes_found
assert FailureClass.OptimalityHallucination in classes_found
assert compound_report.strongest_disposition == "REFUSE"
print("[OK] Exercise 2 verified!")

# Solution 3: 4-Cycle Multi-pass Collapse
U4 = frozenset(product(range(-2, 3), repeat=4))  # 5^4 = 625 candidates
base_ops_4 = [SHAPE(4), PIN_integers(), AntiMorphism("LAND_ZERO_4", "LAND", lambda x: sum(x) == 0)]
chain_4, res_4, _ = collapse(U4, base_ops_4)
demand_4 = DEMAND("prefix=(1,2,-1)", lambda x: x[:3] == (1, 2, -1))
chain_4_final, final_4, verd_4 = collapse(res_4, [demand_4])
assert verd_4 == "LANDED"
(w4,) = list(final_4)
assert w4 == (1, 2, -1, -2)  # 1 + 2 - 1 - 2 = 0
print(f"[OK] Exercise 3 verified! Landed 4-cycle witness: {w4}")

## Summary & Practitioner Takeaways

1. **Three-Layer Sovereignty is Mandatory**: Never let a generative model certify its own output. Keep model weights at Layer 1, decode sampling at Layer 2, and all verification at Layer 3.
2. **Fail-Closed Tooling**: Daily workflow hooks in Claude Code, Codex, and Antigravity swarms must refuse suspect or unverified state immediately.
3. **The 12 Epistemic Failure Classes**: Check for self-certifications, fake receipts, theorem name laundering, and complexity claims deterministically.
4. **Anti-Morphism Collapse**: Verification is an interior operator that shrinks proposal clouds to landed witnesses or honest empty refusals.


> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `00_agentic_symbol_game_quickstart`
- The code checks one finite example. It is not a general proof.
